In [2]:
# 读取所有高质量数据集（score_0.75_to_1.0）并合并为一个dataset对象
from datasets import load_from_disk, concatenate_datasets
import os
# 定义数据集的根目录和目标分数目录
root_dir = '/DATA/disk2/yuhang/.cache/bit_brain_data/chinese-fineweb-classification'
target_score_dir_name = 'score_0.6_to_0.75'

# 用来存储所有加载的数据集
datasets_list = []

print(f"开始加载中等质量数据集 (score_0.6_to_0.75)...")

# 遍历根目录，找到所有的 source_* 文件夹
try:
    for entry in os.scandir(root_dir):
        # 确保它是一个目录并且以 'source_' 开头
        if entry.is_dir() and entry.name.startswith('source_'):
            # 提取 source 的名字
            source_name = entry.name.replace('source_', '')
            
            # 构造高质量数据文件夹的完整路径
            high_score_path = os.path.join(entry.path, target_score_dir_name)
            
            # 检查路径是否存在
            if os.path.exists(high_score_path):
                print(f"  正在加载 '{source_name}' 数据集...")
                try:
                    # 加载数据集
                    source_dataset = load_from_disk(high_score_path)
                    # 添加source信息到数据集中，方便后续分析
                    #source_dataset = source_dataset.add_column("source", [source_name] * len(source_dataset))
                    datasets_list.append(source_dataset)
                    print(f"    成功加载 {len(source_dataset)} 条数据")
                except Exception as e:
                    print(f"    错误: 无法加载 '{source_name}' 数据集: {e}")
            else:
                print(f"  跳过: '{source_name}' - 路径不存在")

    # 合并所有数据集
    if datasets_list:
        print(f"\n正在合并 {len(datasets_list)} 个数据集...")
        dataset = concatenate_datasets(datasets_list)
        print(f"合并完成！总共包含 {len(dataset)} 条高质量数据")
        
        # 显示数据集的基本信息
        print(f"\n数据集信息:")
        print(f"  - 总样本数: {len(dataset):,}")
        print(f"  - 列名: {dataset.column_names}")
        
    #     # 显示各个source的数据分布
    #     if 'source' in dataset.column_names:
    #         source_counts = {}
    #         for item in dataset:
    #             source = item['source']
    #             source_counts[source] = source_counts.get(source, 0) + 1
            
    #         print(f"\n各数据源分布:")
    #         for source, count in sorted(source_counts.items(), key=lambda x: x[1], reverse=True):
    #             percentage = (count / len(dataset)) * 100
    #             print(f"  - {source:<20}: {count:>10,} ({percentage:.2f}%)")
    # else:
    #     print("错误: 没有找到任何可用的数据集")
    #     dataset = None

except FileNotFoundError:
    print(f"错误: 根目录 '{root_dir}' 不存在，请检查路径是否正确。")
    dataset = None

开始加载中等质量数据集 (score_0.6_to_0.75)...
  跳过: 'wanjuan' - 路径不存在
  正在加载 'MiChao' 数据集...
    成功加载 3542042 条数据
  正在加载 'CCI3' 数据集...
    成功加载 28199934 条数据
  正在加载 'IndustryCorpus2' 数据集...
    成功加载 33838292 条数据
  正在加载 'ChineseWebText' 数据集...
    成功加载 18645779 条数据
  跳过: 'WuDao' - 路径不存在
  正在加载 'SkyPile' 数据集...
    成功加载 15615851 条数据
  正在加载 'TeleChat' 数据集...
    成功加载 23836856 条数据

正在合并 6 个数据集...
合并完成！总共包含 123678754 条高质量数据

数据集信息:
  - 总样本数: 123,678,754
  - 列名: ['text', 'score', '__index__', 'source']


In [3]:
import os
import math
from pathlib import Path

def get_dir_size(path):
    """
    计算指定路径下所有文件和子文件夹的总大小。
    这是一个递归函数，意味着它会调用自己来处理子文件夹。
    """
    total_size = 0
    # Path(path).rglob('*') 会遍历目录下的所有内容，包括子文件夹里的
    # 这是一种更现代、更简洁的写法
    for file in Path(path).rglob('*'):
        # 确保我们只计算文件的大小
        if file.is_file():
            total_size += file.stat().st_size
    return total_size

def format_size(size_bytes):
    """
    将字节大小格式化为人类易读的形式 (B, KB, MB, GB, TB)。
    """
    if size_bytes == 0:
        return "0B"
    # 定义大小单位的元组
    size_names = ("B", "KB", "MB", "GB", "TB", "PB", "EB", "ZB", "YB")
    # 使用log计算单位的索引
    i = int(math.floor(math.log(size_bytes, 1024)))
    # 计算转换后的大小
    p = math.pow(1024, i)
    s = round(size_bytes / p, 2)
    return f"{s} {size_names[i]}"

# 1. 定义数据集的根目录
# 根据你的日志，数据保存在这个路径下
root_dir = '/DATA/disk2/yuhang/.cache/bit_brain_data/chinese-fineweb-classification'
# 我们只关心高质量数据集
target_score_dir_name = 'score_0.6_to_0.75'

# 用一个字典来存储每个source和它对应的大小
source_sizes = {}

print(f"开始扫描目录: {root_dir}\n")

# 2. 遍历根目录，找到所有的 source_* 文件夹
# os.scandir 比 os.listdir 更高效，因为它在扫描时就获取了文件信息
try:
    for entry in os.scandir(root_dir):
        # 确保它是一个目录并且以 'source_' 开头
        if entry.is_dir() and entry.name.startswith('source_'):
            # 提取 source 的名字，比如从 'source_CCI3' 提取出 'CCI3'
            source_name = entry.name.replace('source_', '')
            
            # 构造我们感兴趣的高质量数据文件夹的完整路径
            high_score_path = os.path.join(entry.path, target_score_dir_name)
            
            # 3. 检查路径是否存在，然后计算大小
            if os.path.exists(high_score_path):
                print(f"正在计算 '{source_name}' 的大小...")
                # 调用我们写的函数来获取文件夹大小
                size = get_dir_size(high_score_path)
                source_sizes[source_name] = size
            else:
                # 如果某个source没有这个分数段的数据，也打印出来，方便我们知晓
                print(f"  - 警告: 路径 '{high_score_path}' 不存在，跳过。")
except FileNotFoundError:
    print(f"错误: 根目录 '{root_dir}' 不存在，请检查路径是否正确。")

# 4. 计算总大小和各自的比例
total_size = sum(source_sizes.values())

print("\n--- 结果分析 ---")
if total_size == 0:
    print("未能计算任何数据的大小，总大小为 0。请检查目录结构和路径是否正确。")
else:
    # 使用我们写的格式化函数，让结果更易读
    print(f"所有 'score_0.6_to_0.75' 数据的总大小: {format_size(total_size)}\n")
    print("各 source 数据大小及其占比:")
    
    # 按照大小从大到小排序，这样结果看起来更有条理
    sorted_sources = sorted(source_sizes.items(), key=lambda item: item[1], reverse=True)
    
    for source, size in sorted_sources:
        # 计算百分比
        proportion = (size / total_size) * 100
        # 使用 f-string 格式化输出，让表格对齐，更美观
        print(f"  - {source:<20}: {format_size(size):<10} ({proportion:.2f}%)")


开始扫描目录: /DATA/disk2/yuhang/.cache/bit_brain_data/chinese-fineweb-classification

  - 警告: 路径 '/DATA/disk2/yuhang/.cache/bit_brain_data/chinese-fineweb-classification/source_wanjuan/score_0.6_to_0.75' 不存在，跳过。
正在计算 'MiChao' 的大小...
正在计算 'CCI3' 的大小...
正在计算 'IndustryCorpus2' 的大小...
正在计算 'ChineseWebText' 的大小...
  - 警告: 路径 '/DATA/disk2/yuhang/.cache/bit_brain_data/chinese-fineweb-classification/source_WuDao/score_0.6_to_0.75' 不存在，跳过。
正在计算 'SkyPile' 的大小...
正在计算 'TeleChat' 的大小...

--- 结果分析 ---
所有 'score_0.6_to_0.75' 数据的总大小: 568.56 GB

各 source 数据大小及其占比:
  - IndustryCorpus2     : 156.74 GB  (27.57%)
  - CCI3                : 150.58 GB  (26.48%)
  - TeleChat            : 111.07 GB  (19.54%)
  - ChineseWebText      : 75.75 GB   (13.32%)
  - SkyPile             : 59.59 GB   (10.48%)
  - MiChao              : 14.82 GB   (2.61%)


In [6]:
from datasets import load_from_disk
IndustryCorpus2 = load_from_disk("/DATA/disk2/yuhang/.cache/bit_brain_data/chinese-fineweb-classification/source_IndustryCorpus2/score_0.6_to_0.75")

print(IndustryCorpus2)

print(IndustryCorpus2[800])

Dataset({
    features: ['text', 'score', '__index__', 'source'],
    num_rows: 33838292
})
{'text': '9月9日至11日,由中国倡议的国际粮食减损大会在山东济南首次召开.大会发布了"国际粮食减损大会济南倡议".倡议全文如下:\n2021年9月9-11日,来自世界上48个国家,国际组织,企业,非政府组织共150名代表,齐聚中国济南,出席 "国际粮食减损大会",围绕"减少粮食损失浪费,促进世界粮食安全"主题进行深入交流,取得广泛共识.\n我们认为,减少食物损失和浪费是提高粮食安全的重要手段和紧迫的全球性命题.在二十国集团以往有关粮食安全和营养的承诺基础上,围绕粮食减损共同目标,提出如下倡议:\n粮食安全作为全人类的共同命题,是世界和平与发展的重要保障.粮食减损是提高粮食安全和营养水平的必由之路,同每个国家,每个人息息相关.我们呼吁各国树立减损就是增产,降耗就是增收的意识,达成 "减少粮食损失与浪费,促进世界粮食安全与营养"共识.\n针对环境,气候灾害等,提升农业基础设施水平,加强粮食综合生产能力建设,提升粮食系统供给韧性.各国家和地区,组织,机构等积极行动起来,加强气候变化对农业影响的评估和应对措施,制定适应和减缓气候变化的行动方案.在食物生产,流通,消费全过程中积极应对气候变化影响,进一步增强农业适应气候变化的能力.\n把生态环境保护,创新和可持续集约化作为农业生产的基本前提和刚性约束,推动创新,构建可持续的生产体系,推行资源节约,环境友好,生态循环的生产方式.塑料材料等生产资料应尽可能回收利用.在良种选育,绿色防控,科学施肥,精准灌溉,高效管理等方面加大工作力度,提高生产者的专业技能,提升粮食生产机械化效率,切实做到减损增效.\n我们将积极运用新技术,新设备,新工艺,加强集约,可持续,低碳的现代化粮食产后服务体系建设,为农户提供科学储粮的技术培训和服务,重点加强小农户储粮新装具的推广和使用,提升粮食品质.在技术研发,标准规范,投资引导等方面,推动仓储,运输,加工等环节有机融合,有效衔接,实现产后全过程,系统化节粮减损,建设"无形良田",实现"无地增产".\n鼓励政府,企业,组织等开展系列活动,引导民众树立节约意识,养成良好习惯.加大科普宣传力度,从娃娃抓起,开展食育进

In [9]:
# 只分析前10000条数据以快速了解数据分布
sample_size = 10000
print(f"正在分析前 {sample_size} 条数据的text字段长度分布...")

# 分析wanjuan数据集中text字段的长度分布
import numpy as np

print("正在分析wanjuan数据集中text字段的长度分布...")

# 计算所有text的长度
text_lengths = []
print("正在计算每条数据的text长度...")

# 遍历数据集计算每条text的长度
for i, example in enumerate(IndustryCorpus2[:sample_size]):
    text_length = len(example['text'])
    text_lengths.append(text_length)
    
    # 每处理10000条数据显示一次进度
    if (i + 1) % 10000 == 0:
        print(f"  已处理 {i + 1} 条数据...")

# 转换为numpy数组便于统计分析
text_lengths = np.array(text_lengths)

print(f"\n--- wanjuan数据集text长度统计分析 ---")
print(f"数据总条数: {len(text_lengths):,}")
print(f"text长度均值: {text_lengths.mean():.2f} 字符")
print(f"text长度中位数: {np.median(text_lengths):.2f} 字符")
print(f"text长度标准差: {text_lengths.std():.2f} 字符")
print(f"text最短长度: {text_lengths.min()} 字符")
print(f"text最长长度: {text_lengths.max():,} 字符")

# 计算一些百分位数，了解数据分布
percentiles = [25, 50, 75, 90, 95, 99]
print(f"\ntext长度百分位数分布:")
for p in percentiles:
    value = np.percentile(text_lengths, p)
    print(f"  {p}%分位数: {value:.0f} 字符")

# 统计不同长度区间的数据分布
print(f"\ntext长度区间分布:")
bins = [0, 100, 500, 1000, 2000, 5000, 10000, float('inf')]
bin_labels = ['0-100', '100-500', '500-1K', '1K-2K', '2K-5K', '5K-10K', '10K+']

for i in range(len(bins)-1):
    count = np.sum((text_lengths >= bins[i]) & (text_lengths < bins[i+1]))
    percentage = (count / len(text_lengths)) * 100
    print(f"  {bin_labels[i]:<8}: {count:>8,} 条 ({percentage:>5.2f}%)")


正在分析前 10000 条数据的text字段长度分布...
正在分析wanjuan数据集中text字段的长度分布...
正在计算每条数据的text长度...


TypeError: string indices must be integers, not 'str'

In [15]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B",
                                          use_fast = True)
tokenizer

Qwen2TokenizerFast(name_or_path='Qwen/Qwen3-0.6B', vocab_size=151643, model_max_length=131072, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>']}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized

In [11]:
# 读取所有高质量数据集（score_0.75_to_1.0）并合并为一个dataset对象
from datasets import load_from_disk, concatenate_datasets
import os
# 定义数据集的根目录和目标分数目录
root_dir = '/DATA/disk2/yuhang/.cache/bit_brain_data/chinese-fineweb-classification'
target_score_dir_name = 'score_0.6_to_0.75'

# 用来存储所有加载的数据集
datasets_list = []

print(f"开始加载高质量数据集 (score_0.6_to_0.75)...")

# 遍历根目录，找到所有的 source_* 文件夹
try:
    for entry in os.scandir(root_dir):
        # 确保它是一个目录并且以 'source_' 开头
        if entry.is_dir() and entry.name.startswith('source_'):
            # 提取 source 的名字
            source_name = entry.name.replace('source_', '')
            
            # 构造高质量数据文件夹的完整路径
            high_score_path = os.path.join(entry.path, target_score_dir_name)
            
            # 检查路径是否存在
            if os.path.exists(high_score_path):
                print(f"  正在加载 '{source_name}' 数据集...")
                try:
                    # 加载数据集
                    source_dataset = load_from_disk(high_score_path)
                    # 添加source信息到数据集中，方便后续分析
                    #source_dataset = source_dataset.add_column("source", [source_name] * len(source_dataset))
                    datasets_list.append(source_dataset)
                    print(f"    成功加载 {len(source_dataset)} 条数据")
                except Exception as e:
                    print(f"    错误: 无法加载 '{source_name}' 数据集: {e}")
            else:
                print(f"  跳过: '{source_name}' - 路径不存在")

    # 合并所有数据集
    if datasets_list:
        print(f"\n正在合并 {len(datasets_list)} 个数据集...")
        dataset = concatenate_datasets(datasets_list)
        print(f"合并完成！总共包含 {len(dataset)} 条高质量数据")
        
        # 显示数据集的基本信息
        print(f"\n数据集信息:")
        print(f"  - 总样本数: {len(dataset):,}")
        print(f"  - 列名: {dataset.column_names}")
        
    #     # 显示各个source的数据分布
    #     if 'source' in dataset.column_names:
    #         source_counts = {}
    #         for item in dataset:
    #             source = item['source']
    #             source_counts[source] = source_counts.get(source, 0) + 1
            
    #         print(f"\n各数据源分布:")
    #         for source, count in sorted(source_counts.items(), key=lambda x: x[1], reverse=True):
    #             percentage = (count / len(dataset)) * 100
    #             print(f"  - {source:<20}: {count:>10,} ({percentage:.2f}%)")
    # else:
    #     print("错误: 没有找到任何可用的数据集")
    #     dataset = None

except FileNotFoundError:
    print(f"错误: 根目录 '{root_dir}' 不存在，请检查路径是否正确。")
    dataset = None

开始加载高质量数据集 (score_0.6_to_0.75)...
  跳过: 'wanjuan' - 路径不存在
  正在加载 'MiChao' 数据集...
    成功加载 3542042 条数据
  正在加载 'CCI3' 数据集...
    成功加载 28199934 条数据
  正在加载 'IndustryCorpus2' 数据集...
    成功加载 33838292 条数据
  正在加载 'ChineseWebText' 数据集...
    成功加载 18645779 条数据
  跳过: 'WuDao' - 路径不存在
  正在加载 'SkyPile' 数据集...
    成功加载 15615851 条数据
  正在加载 'TeleChat' 数据集...
    成功加载 23836856 条数据

正在合并 6 个数据集...
合并完成！总共包含 123678754 条高质量数据

数据集信息:
  - 总样本数: 123,678,754
  - 列名: ['text', 'score', '__index__', 'source']


In [14]:
dataset

Dataset({
    features: ['text', 'score', '__index__', 'source'],
    num_rows: 123678754
})

#### 对中等分数的industryCorpus2数据集进行分词处理

In [17]:
from datasets import load_from_disk
IndustryCorpus2 = load_from_disk("/DATA/disk2/yuhang/.cache/bit_brain_data/chinese-fineweb-classification/source_IndustryCorpus2/score_0.6_to_0.75")

print(IndustryCorpus2)

print(IndustryCorpus2[800])

Dataset({
    features: ['text', 'score', '__index__', 'source'],
    num_rows: 33838292
})
{'text': '9月9日至11日,由中国倡议的国际粮食减损大会在山东济南首次召开.大会发布了"国际粮食减损大会济南倡议".倡议全文如下:\n2021年9月9-11日,来自世界上48个国家,国际组织,企业,非政府组织共150名代表,齐聚中国济南,出席 "国际粮食减损大会",围绕"减少粮食损失浪费,促进世界粮食安全"主题进行深入交流,取得广泛共识.\n我们认为,减少食物损失和浪费是提高粮食安全的重要手段和紧迫的全球性命题.在二十国集团以往有关粮食安全和营养的承诺基础上,围绕粮食减损共同目标,提出如下倡议:\n粮食安全作为全人类的共同命题,是世界和平与发展的重要保障.粮食减损是提高粮食安全和营养水平的必由之路,同每个国家,每个人息息相关.我们呼吁各国树立减损就是增产,降耗就是增收的意识,达成 "减少粮食损失与浪费,促进世界粮食安全与营养"共识.\n针对环境,气候灾害等,提升农业基础设施水平,加强粮食综合生产能力建设,提升粮食系统供给韧性.各国家和地区,组织,机构等积极行动起来,加强气候变化对农业影响的评估和应对措施,制定适应和减缓气候变化的行动方案.在食物生产,流通,消费全过程中积极应对气候变化影响,进一步增强农业适应气候变化的能力.\n把生态环境保护,创新和可持续集约化作为农业生产的基本前提和刚性约束,推动创新,构建可持续的生产体系,推行资源节约,环境友好,生态循环的生产方式.塑料材料等生产资料应尽可能回收利用.在良种选育,绿色防控,科学施肥,精准灌溉,高效管理等方面加大工作力度,提高生产者的专业技能,提升粮食生产机械化效率,切实做到减损增效.\n我们将积极运用新技术,新设备,新工艺,加强集约,可持续,低碳的现代化粮食产后服务体系建设,为农户提供科学储粮的技术培训和服务,重点加强小农户储粮新装具的推广和使用,提升粮食品质.在技术研发,标准规范,投资引导等方面,推动仓储,运输,加工等环节有机融合,有效衔接,实现产后全过程,系统化节粮减损,建设"无形良田",实现"无地增产".\n鼓励政府,企业,组织等开展系列活动,引导民众树立节约意识,养成良好习惯.加大科普宣传力度,从娃娃抓起,开展食育进

In [18]:
def tokenize_function(examples):
    return tokenizer(examples['text'],
                     max_length=2048,
                     truncation=True,
                     padding="max_length")

tokenizer_dataset = IndustryCorpus2.map(tokenize_function,
                                num_proc=40,
                                batched=True, 
                                remove_columns=['text']) #! 移除原始文本节省内存

Map (num_proc=40): 100%|██████████| 33838292/33838292 [34:05<00:00, 16541.57 examples/s]


In [19]:
tokenizer_dataset.format

{'type': None,
 'format_kwargs': {},
 'columns': ['score', '__index__', 'source', 'input_ids', 'attention_mask'],
 'output_all_columns': False}

In [20]:
tokenizer_dataset.save_to_disk("/DATA/disk2/yuhang/.cache/bit_brain_data/step3_industry_mid_tokenizer_data",
                               max_shard_size = "2024MB")

Saving the dataset (172/172 shards): 100%|██████████| 33838292/33838292 [04:01<00:00, 140341.00 examples/s]


#### 对中等分数的CCI3数据集进行分词处理

In [24]:
from datasets import load_from_disk
CCI3 = load_from_disk("/DATA/disk2/yuhang/.cache/bit_brain_data/chinese-fineweb-classification/source_CCI3/score_0.6_to_0.75")

print(CCI3)

print(CCI3[800])

Dataset({
    features: ['text', 'score', '__index__', 'source'],
    num_rows: 28199934
})
{'text': '2022 年开年以来，信贷政策利好不断。\n在 2021 年两次全面降准以及 1 月 17 日下调 MLF 利率和逆回购汇率后， 1 月 20 日，新一期贷款市场报价利率( LPR )出炉。\n央行本周内三次明确释放“宽货币”信号，开年以来两次降息对于提振市场信心有着至关重要的作用，货币政策窗口期已至。\n本文 约 2535 字 阅读 需要 5 min\nLPR调整来看，本次有两大特别之处。\n首先，1年期LPR出现罕见的连续调整。\n最直观的影响是购房者按揭，但又不仅只影响按揭，LPR下行将带动个人住房贷款利率降低，同时降低企业融资成本。\n在当前的利率体系中，LPR属市场利率范畴，但因其对市场主体融资成本特别是银行贷款成本具有重大影响，因此具有较强的政策信号意义。\n而5年期LPR下调对降低全社会融资成本的覆盖面更大，包括个人按揭贷款和企业中长期贷款。\n这意味着，信贷政策由此前的居民端(提高个人按揭贷款额度、缩短放贷周期、下调房贷利率等)逐步惠及企业端(降低企业融资成本)，避免因局部企业“爆雷”造成系统性危机，进而全面重铸房地产行业信心。\n图:2019年8月-2022年1月市场报价利率LPR\nLPR利率非对称下调意在“稳增长”\n实际上，基于近期央行多次表态，以及本周内MLF超量续作且下调政策利率，而从历次调整来看，LPR均会与MLF利率、逆回购操作利率同步调整。\n1月18日，对于LPR是否会同步调整的问题，央行货币政策司司长孙国峰在国新办发布会上表示，LPR报价行报价时综合考虑自身资金成本、风险溢价和市场供求等因素，LPR会及时充分反映市场利率变化，引导企业贷款利率下行，有力推动降低企业综合融资成本。\n实际上， 市场对本次LPR利率下降已有充分预期 ，关注的焦点在于长短期LPR利率是否对称下调，以及调整幅度。\n本次长短期LPR利率非对称下调，加之上一次12月份的调整，1年期LPR较去年4月累计下调15BP，5年期LPR下调5BP，两者利差走阔10BP。\n图:2019年8月-2022年1月MLF、LPR利率历次变动情况\n激发市场主体融资需求，购房

In [25]:
def tokenize_function(examples):
    return tokenizer(examples['text'],
                     max_length=2048,
                     truncation=True,
                     padding="max_length")

CCI3_tokenizer_dataset = CCI3.map(tokenize_function,
                                num_proc=40,
                                batched=True, 
                                remove_columns=['text']) #! 移除原始文本节省内存

Map (num_proc=40): 100%|██████████| 28199934/28199934 [32:29<00:00, 14466.18 examples/s]


In [27]:
CCI3_tokenizer_dataset.save_to_disk("/DATA/disk2/yuhang/.cache/bit_brain_data/step3_cci3_mid_tokenizer_data",
                               max_shard_size = "2024MB")

Saving the dataset (144/144 shards): 100%|██████████| 28199934/28199934 [04:55<00:00, 95516.77 examples/s] 
